# 11 — Judging pass: StrongREJECT + WildGuard on all nine stages

**Why this run exists.** Every cross-stage behavioural number currently rests on
the frozen regex classifier alone, because the original judge run used
`--scope confirmatory` and scored only the preregistered populations. On the 208
rows where all three scorers exist (quadrant C of M2 and M3), the regex
classifier agrees with WildGuard at **Cohen's kappa = 0.29** — it finds 1 refusal
in 104 rows where WildGuard finds 42, because it only matches templated refusals
("I cannot", "I'm sorry") and the models refuse these prompts in plain language.

So this pass is **load-bearing, not confirmatory**. It scores the ~5,678
behaviour rows that were never scored.

**No generation happens here.** The responses already exist. This only runs the
two judge models over them.

- Expected wall clock: **~80–90 min** (two model loads + ~5,700 rows x 2 judges)
- Needs: a T4 or better, and the previous judge file in Drive for `--resume-from`
- Session budget: comfortably inside one Colab session

**Run the cells in order. Cell 5 is a preflight that stops early if the judges
cannot load — that failure mode previously produced three complete-looking but
empty output files.**


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin the exact commit

**Set `PINNED_COMMIT` to the SHA you pushed before starting this session.** The
WP-2 fail-closed guard must be present, or a judge load failure will silently
write a complete-looking file again.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'main'
PINNED_COMMIT = 'PASTE_THE_SHA_HERE'   # <-- set this

assert PINNED_COMMIT != 'PASTE_THE_SHA_HERE', (
    "Set PINNED_COMMIT to the commit you pushed. Locally:\n"
    "  git rev-parse HEAD")

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert commit == PINNED_COMMIT, f"wrong commit: {commit} != {PINNED_COMMIT}"
print("checked out", commit)

# the WP-2 guard must exist in this checkout
from importlib import util as _u
assert _u.find_spec("src.analysis.behavioral_judges")
import inspect, src.analysis.behavioral_judges as bj
assert "fail_closed" in inspect.signature(bj.run_judges).parameters, (
    "This commit predates the WP-2 fail-closed guard. Push it first.")
print("fail-closed guard present")

## 3. Dependencies and GPU

In [ ]:
!pip -q install -r requirements.txt
# Colab preinstalls torchao; it breaks transformers on quantized loads (see 04b).
!pip uninstall -y torchao || true
!nvidia-smi

## 4. Bind results/ and the HF cache to Drive

In [ ]:
from src.colab_persist import bind, status_line
info = bind()
print(status_line(info))

## 5. PREFLIGHT — stop here if the judges do not load

This is the cell that would have prevented the three orphaned 15 MB files. If
either judge reports `False`, **do not continue**: fix the environment first.

In [ ]:
from src.analysis.behavioral_judges import LazyModelJudge

sr = LazyModelJudge('strong_reject', 'dsbowen/strong_reject', load_4bit=True)
ok_sr = sr.try_load()
print('strong_reject:', ok_sr, '' if ok_sr else sr.load_error)
if ok_sr:
    sr.unload()

wg = LazyModelJudge('wildguard', 'allenai/wildguard', load_4bit=True)
ok_wg = wg.try_load()
print('wildguard    :', ok_wg, '' if ok_wg else wg.load_error)
if ok_wg:
    wg.unload()

assert ok_sr and ok_wg, (
    "A judge failed to load. STOP. Fix the environment before running the pass "
    "- continuing would burn ~90 min and produce nothing.")
print("\npreflight OK")

## 6. Verify the inputs are present

`--resume-from` needs the previous judge file (75 MB, gitignored, so it comes
from Drive not GitHub). If this cell fails, upload it to
`MyDrive/dpo_v2/results/behavioral_judges/` and re-run.

In [ ]:
import json
from pathlib import Path

RESUME = Path("results/behavioral_judges/behavioral_judges_v2_20260907T043919Z.json")
MANIFEST = Path("results/manifests/consolidated_judge.json")

assert RESUME.exists(), (
    f"missing {RESUME}\n"
    "Upload it to MyDrive/dpo_v2/results/behavioral_judges/ and re-run this cell.")
assert MANIFEST.exists(), f"missing {MANIFEST}"

prior = json.loads(RESUME.read_text(encoding="utf-8"))
n_prior = len(prior["records"])
print(f"resume file : {n_prior:,} records")
print(f"judge_status: {prior.get('judge_status')}")
assert n_prior == 30798, f"expected 30,798 prior records, got {n_prior:,}"
assert all(v == "scored" for v in prior.get("judge_status", {}).values()), (
    "the resume file itself records a judge failure - do not build on it")

# how many behaviour rows still need scoring
behav = [r for r in prior["records"] if r.get("condition", "").endswith("_behavior")]
todo = [r for r in behav
        if (r.get("strong_reject") or {}).get("judge_status") != "scored"]
print(f"behaviour rows: {len(behav):,}   unscored by StrongREJECT: {len(todo):,}")

## 7. The judging pass

`--scope all` is essential: the default `confirmatory` scope is what excluded
these rows in the first place. The **coverage contract** makes the run fail
loudly if any declared cell is still unscored when it finishes.

In [ ]:
import time
from src.analysis.behavioral_judges import run_judges
from src.common.stages import ALL_STAGES

# every stage's behaviour rows, every quadrant, under both LLM judges
CONTRACT = {
    scorer: [(f"{stage}_behavior", None) for stage in ALL_STAGES]
    for scorer in ("strong_reject", "wildguard")
}
print(f"contract: {len(CONTRACT['strong_reject'])} conditions x 2 scorers")

t0 = time.time()
out = run_judges(
    "results/manifests/consolidated_judge.json",
    out_dir="results/behavioral_judges",
    run_live=True,
    scope="all",
    resume_from="results/behavioral_judges/behavioral_judges_v2_20260907T043919Z.json",
    load_4bit=True,
    allow_download=True,
    fail_closed=True,
    coverage_contract=CONTRACT,
)
print(f"\nwrote {out}  in {(time.time() - t0) / 60:.1f} min")

## 8. Verify coverage actually improved

In [ ]:
import json
from pathlib import Path

payload = json.loads(Path(out).read_text(encoding="utf-8"))
print("judge_status:", payload["judge_status"])
print("carried forward:", payload.get("carried_forward"))

from collections import defaultdict
cov = defaultdict(lambda: defaultdict(int))
for r in payload["records"]:
    c = r.get("condition", "")
    if not c.endswith("_behavior"):
        continue
    cov[c]["n"] += 1
    for s in ("regex", "strong_reject", "wildguard"):
        if (r.get(s) or {}).get("judge_status") == "scored":
            cov[c][s] += 1

print(f"\n{'condition':24s} {'rows':>6s} {'regex':>7s} {'SR':>7s} {'WG':>7s}")
for c in sorted(cov):
    v = cov[c]
    print(f"{c:24s} {v['n']:6d} {v['regex']:7d} {v['strong_reject']:7d} {v['wildguard']:7d}")

incomplete = [c for c, v in cov.items()
              if v["strong_reject"] < v["n"] or v["wildguard"] < v["n"]]
assert not incomplete, f"still incomplete: {incomplete}"
print("\nAll nine behaviour conditions fully scored by both judges.")

## 9. Bring back to the analysis session

The new judge file is in Drive under `dpo_v2/results/behavioral_judges/`. Report
back:

1. the **filename** of the new judge file (it is timestamped),
2. the **coverage table** printed by cell 8,
3. the **wall clock** printed by cell 7.

The cross-stage behavioural matrix, the 2x2 contrasts and the selectivity
results will then be recomputed under all three scorers, and the regex-only
numbers become a sensitivity row rather than the headline.

In [ ]:
print("new judge file:")
!ls -la results/behavioral_judges/*.json | tail -3